# 03. Model Training — Google Colab GPU (Standalone)

This is a self-contained notebook to train the Indian Sign Language gesture models (BiLSTM and Transformer) on Google Colab using a T4 GPU. You **do not** need to upload the entire codebase to run this. You only need the extracted landmarks.

### Instructions:
1. In Google Drive, create a folder named `ISL-Speak`.
2. Upload your local `data/splits` folder (which contains `X_train.npz`, `y_train.npz`, etc.) into that `ISL-Speak` folder.
3. Go to **Runtime > Change runtime type** in Colab and select **T4 GPU**.
4. Run the cells below.

In [ ]:
# 1. Mount Google Drive & Environment Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define paths (Update this if you uploaded the splits folder elsewhere)
SPLITS_DIR = '/content/drive/MyDrive/ISL-Speak/splits'
CHECKPOINT_DIR = '/content/drive/MyDrive/ISL-Speak/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# 2. Hyperparameters Configuration
SEQ_LEN = 45
TOTAL_FEATURES = 258
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 60
PATIENCE = 10

In [ ]:
# 3. Load Dataset
def load_dataset(splits_dir):
    train_data = np.load(os.path.join(splits_dir, "X_train.npz"))
    val_data = np.load(os.path.join(splits_dir, "X_val.npz"))

    X_train, y_train = train_data["X"], train_data["y"]
    X_val, y_val = val_data["X"], val_data["y"]

    mapping_json = os.path.join(splits_dir, "class_index_to_label.json")
    if os.path.exists(mapping_json):
        with open(mapping_json, "r") as f:
            class_mapping = json.load(f)
        num_classes = len(class_mapping)
    else:
        num_classes = len(np.unique(y_train))
        
    print(f"Loaded Data:\n X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f" X_val: {X_val.shape}, y_val: {y_val.shape}")
    print(f" Number of classes: {num_classes}")

    return X_train, y_train, X_val, y_val, num_classes

X_train, y_train, X_val, y_val, NUM_CLASSES = load_dataset(SPLITS_DIR)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 4. Define BiLSTM Model
class SignLSTMClassifier(nn.Module):
    def __init__(self, num_features=TOTAL_FEATURES, num_classes=NUM_CLASSES, hidden_size=128, num_layers=2, dropout=0.3):
        super(SignLSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=num_features, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        forward_final = h_n[-2, :, :]
        backward_final = h_n[-1, :, :]
        out = torch.cat((forward_final, backward_final), dim=1)
        out = self.dropout(out)
        return self.fc(out)

In [ ]:
# 5. Define Transformer Model
class SignTransformerClassifier(nn.Module):
    def __init__(self, num_features=TOTAL_FEATURES, num_classes=NUM_CLASSES, d_model=128, nhead=8, num_layers=4, dim_feedforward=256, dropout=0.3, seq_len=SEQ_LEN):
        super(SignTransformerClassifier, self).__init__()
        self.input_projection = nn.Linear(num_features, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.normal_(self.pos_embedding, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        h = self.input_projection(x)
        h = h + self.pos_embedding[:, :seq_len, :]
        h_out = self.transformer_encoder(h)
        h_pooled = h_out.mean(dim=1)
        h_pooled = self.dropout(h_pooled)
        return self.fc(h_pooled)

In [ ]:
# 6. Universal Training Loop
def train_model(model, model_name):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best_val_acc = 0.0
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"best_{model_name}_model.pt")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_train += (preds == targets).sum().item()
            total_train += targets.size(0)

        epoch_train_loss = running_loss / total_train
        epoch_train_acc = correct_train / total_train

        model.eval()
        val_running_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_running_loss += loss.item() * inputs.size(0)
                preds = torch.argmax(outputs, dim=1)
                correct_val += (preds == targets).sum().item()
                total_val += targets.size(0)

        epoch_val_loss = val_running_loss / total_val
        epoch_val_acc = correct_val / total_val
        scheduler.step(epoch_val_acc)

        history["train_loss"].append(epoch_train_loss)
        history["val_loss"].append(epoch_val_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_acc"].append(epoch_val_acc)

        print(f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:.2f}%")

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_type": model_name,
                "state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": best_val_acc,
                "num_classes": NUM_CLASSES,
            }, checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping triggered after {epoch} epochs.")
                break
                
    print(f"Best {model_name} Validation Accuracy: {best_val_acc*100:.2f}%")
    print(f"Best model saved to {checkpoint_path}")
    return model, history

In [ ]:
# 7. Train BiLSTM
print("=== Training BiLSTM Model ===")
bilstm_model = SignLSTMClassifier()
_, bilstm_history = train_model(bilstm_model, "lstm")

In [ ]:
# 8. Train Transformer
print("\n=== Training Transformer Model ===")
transformer_model = SignTransformerClassifier()
_, transformer_history = train_model(transformer_model, "transformer")

In [ ]:
# 9. Plot Training Curves
def plot_history(history, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(history["train_loss"], label="Train Loss")
    ax1.plot(history["val_loss"], label="Val Loss")
    ax1.set_title(f"{model_name} Loss Curves")
    ax1.legend()
    
    ax2.plot([acc * 100 for acc in history["train_acc"]], label="Train Acc (%)")
    ax2.plot([acc * 100 for acc in history["val_acc"]], label="Val Acc (%)")
    ax2.set_title(f"{model_name} Accuracy Curves")
    ax2.legend()
    plt.show()

plot_history(bilstm_history, "BiLSTM")
plot_history(transformer_history, "Transformer")